# TAM Ingestion Walkthrough

This notebook walks through the entire Excel-to-Queryable-Data ingestion process step by step.

**What this notebook does:**
1. Lists all raw Excel files in your S3 bucket
2. Lets you select which file(s) to process
3. Shows every step of ingestion with full visibility into inputs, outputs, and LLM calls
4. Saves coordinates to config so you don't have to re-enter them

---

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import json
import yaml
import boto3
import pandas as pd
from pathlib import Path
from datetime import datetime
from IPython.display import display, HTML, Markdown

# Load environment
from dotenv import load_dotenv
load_dotenv('../.env')

# Import pipeline components
from config.settings import Settings, get_settings, reset_settings
from ingestion.excel_reader import read_rectangle, list_sheets, detect_data_range
from ingestion.column_profiler import profile_columns
from ingestion.athena_loader import create_athena_table, save_card
from models.table_card import TableCard, ColumnProfile, DataQuality, EntityProfile, CrossReference, QueryPattern
from models.serialization import serialize_card

# Reset and load fresh settings
reset_settings()
settings = get_settings()

print(f"Storage Mode: {settings.storage_mode}")
print(f"AWS Region: {settings.aws_region}")
print(f"S3 Raw Bucket: {settings.s3_bucket_raw}")
print(f"S3 Processed Bucket: {settings.s3_bucket_processed}")
print(f"Athena Database: {settings.athena_database}")
print(f"LLM Provider: {settings.llm_provider}")

---
## Step 1: List S3 Files & Generate Configs

Lists all Excel files in S3 and auto-generates skeleton configs for any new files.
- Files already in `table_configs.yaml` are shown as configured
- New files get skeleton configs printed - just edit the coordinates

In [ ]:
CONFIG_PATH = Path('../config/table_configs.yaml')

def load_table_configs() -> dict:
    """Load table configurations from YAML."""
    if CONFIG_PATH.exists():
        with open(CONFIG_PATH) as f:
            return yaml.safe_load(f) or {}
    return {}

def save_table_configs(configs: dict):
    """Save table configurations to YAML."""
    with open(CONFIG_PATH, 'w') as f:
        yaml.dump(configs, f, default_flow_style=False, sort_keys=False)

def list_raw_excel_files(bucket: str, prefix: str = "") -> pd.DataFrame:
    """List all Excel files in the raw S3 bucket."""
    s3 = boto3.client('s3', region_name=settings.aws_region)
    
    files = []
    paginator = s3.get_paginator('list_objects_v2')
    
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get('Contents', []):
            key = obj['Key']
            if key.endswith(('.xlsx', '.xls')):
                files.append({
                    's3_key': key,
                    'filename': key.split('/')[-1],
                    'size_kb': round(obj['Size'] / 1024, 1),
                    'last_modified': obj['LastModified'].strftime('%Y-%m-%d %H:%M'),
                })
    
    return pd.DataFrame(files)

def generate_config_key(s3_key: str) -> str:
    """Generate a config key from S3 key (e.g., 'data/sales.xlsx' -> 'sales')"""
    filename = s3_key.split('/')[-1]
    name = filename.rsplit('.', 1)[0]  # Remove extension
    return name.lower().replace(' ', '_').replace('-', '_')

# Load existing configs
table_configs = load_table_configs()
print(f"Loaded {len(table_configs)} existing configurations")

# List S3 files
raw_files_df = list_raw_excel_files(settings.s3_bucket_raw)
print(f"Found {len(raw_files_df)} Excel files in s3://{settings.s3_bucket_raw}/\n")
display(raw_files_df)

# Generate skeleton configs for files not already configured
existing_s3_keys = {cfg.get('s3_key') for cfg in table_configs.values()}
new_configs = {}

for _, row in raw_files_df.iterrows():
    s3_key = row['s3_key']
    if s3_key not in existing_s3_keys:
        config_key = generate_config_key(s3_key)
        # Ensure unique key
        base_key = config_key
        counter = 1
        while config_key in table_configs or config_key in new_configs:
            config_key = f"{base_key}_{counter}"
            counter += 1
        
        new_configs[config_key] = {
            's3_key': s3_key,
            'tables': [
                {
                    'sheet': 'Sheet1',  # EDIT: actual sheet name
                    'table_id': config_key,  # EDIT: unique table ID
                    'start_cell': 'A1',  # EDIT: where headers start
                    'end_cell': None,  # EDIT: end cell or null for auto-detect
                },
                # Add more tables if multiple tables on same sheet:
                # {
                #     'sheet': 'Sheet1',  # same sheet
                #     'table_id': f'{config_key}_table2',
                #     'start_cell': 'A50',
                #     'end_cell': 'K100',
                # },
            ]
        }

if new_configs:
    print(f"\n{'='*60}")
    print(f"Generated {len(new_configs)} NEW skeleton configs for unconfigured files.")
    print("Edit the coordinates below, then run the next cell to save.")
    print(f"{'='*60}\n")
    print(yaml.dump(new_configs, default_flow_style=False, sort_keys=False))
else:
    print("\nAll S3 files already have configurations.")

---
## Step 2: Edit & Save New Configs (if needed)

If Step 1 generated skeleton configs for new files:
1. Copy the YAML output from above
2. Paste it into `new_configs_to_save` below  
3. Edit the sheet names and coordinates
4. Run the cell to save to `table_configs.yaml`

In [ ]:
# ============================================================
# EDIT NEW CONFIGS HERE
# ============================================================
# Copy the skeleton configs from above, edit the coordinates,
# then run this cell to save them to table_configs.yaml
#
# Multiple tables on same sheet? Just add more entries with same 'sheet' value:
#   tables:
#     - sheet: "Data"
#       table_id: "customers"
#       start_cell: "A1"
#       end_cell: "K50"
#     - sheet: "Data"          # same sheet!
#       table_id: "orders"
#       start_cell: "A55"
#       end_cell: "M100"
# ============================================================

# Paste and edit your new configs here:
new_configs_to_save = {
    # Example:
    # 'my_sales_data': {
    #     's3_key': 'sales_report.xlsx',
    #     'tables': [
    #         {
    #             'sheet': 'Q1 Data',      # <- actual sheet name
    #             'table_id': 'sales_q1',  # <- unique table ID
    #             'start_cell': 'B3',      # <- where headers start
    #             'end_cell': 'K150',      # <- or None for auto-detect
    #         },
    #         {
    #             'sheet': 'Q1 Data',      # <- same sheet, different table
    #             'table_id': 'returns_q1',
    #             'start_cell': 'B160',
    #             'end_cell': 'F200',
    #         },
    #     ]
    # },
}

# Save if there are configs to save
if new_configs_to_save:
    table_configs.update(new_configs_to_save)
    save_table_configs(table_configs)
    print(f"Saved {len(new_configs_to_save)} new configurations!")
    print(f"Total configs now: {len(table_configs)}")
else:
    print("No new configs to save. Edit new_configs_to_save above if needed.")

---
## Step 3: Select File & Table to Process

Choose a config and which table to process. Each config can have multiple tables (even on the same sheet).

In [ ]:
# ===========================================
# CONFIGURE THIS CELL
# ===========================================

# Option 1: Use a pre-configured file
USE_CONFIG = "financial_sample"  # Config name from table_configs.yaml
TABLE_INDEX = 0  # Which table to process (0 = first table)

# Option 2: Configure manually
MANUAL_CONFIG = {
    's3_key': 'your_file.xlsx',
    'tables': [
        {
            'sheet': 'Sheet1',
            'table_id': 'your_table_id',
            'start_cell': 'A1',
            'end_cell': None,  # None = auto-detect
        }
    ]
}

# ===========================================

if USE_CONFIG and USE_CONFIG in table_configs:
    selected_config = table_configs[USE_CONFIG]
    config_name = USE_CONFIG
    print(f"Using pre-configured: {USE_CONFIG}")
else:
    selected_config = MANUAL_CONFIG
    config_name = 'manual'
    print("Using manual configuration")

# Show all tables in this config
print(f"\nTables in this config ({len(selected_config.get('tables', []))} total):")
for i, t in enumerate(selected_config.get('tables', [])):
    marker = " <-- SELECTED" if i == TABLE_INDEX else ""
    print(f"  [{i}] {t.get('table_id', 'unnamed')} (sheet: {t.get('sheet')}, {t.get('start_cell')} -> {t.get('end_cell') or 'auto'}){marker}")

# Select the specific table
selected_table = selected_config['tables'][TABLE_INDEX]
print(f"\nSelected table config:")
print(json.dumps(selected_table, indent=2, default=str))

---
## Step 4: Download File from S3

Download the selected Excel file to a local temp location for processing.

In [ ]:
import tempfile

s3_key = selected_config['s3_key']
local_temp_dir = Path(tempfile.mkdtemp())
local_file_path = local_temp_dir / s3_key.split('/')[-1]

print(f"Downloading: s3://{settings.s3_bucket_raw}/{s3_key}")
print(f"To: {local_file_path}")

s3 = boto3.client('s3', region_name=settings.aws_region)
s3.download_file(settings.s3_bucket_raw, s3_key, str(local_file_path))

file_size = local_file_path.stat().st_size / 1024
print(f"\nDownloaded: {file_size:.1f} KB")

---
## Step 5: Explore Excel File Structure

List sheets and auto-detect data ranges.

In [ ]:
# List all sheets
sheets = list_sheets(str(local_file_path))
print(f"Sheets in workbook: {sheets}\n")

# Detect data ranges for each sheet
print("Detected data ranges:")
print("=" * 60)
for sheet in sheets:
    try:
        start, end = detect_data_range(str(local_file_path), sheet)
        print(f"  {sheet}: {start} -> {end}")
    except Exception as e:
        print(f"  {sheet}: Error - {e}")

---
## Step 6: Read Excel Data

Read the data from the configured coordinates.

In [ ]:
# Get table details from selected_table (set in Step 3)
sheet_name = selected_table['sheet']
table_id = selected_table.get('table_id') or f"{config_name}_table"
start_cell = selected_table.get('start_cell', 'A1')
end_cell = selected_table.get('end_cell')  # None = auto-detect

print(f"Reading table: {table_id}")
print(f"Sheet: {sheet_name}")
print(f"Start cell: {start_cell}")
print(f"End cell: {end_cell or 'Auto-detect'}")
print("=" * 60)

# Read the data
df, header_metadata = read_rectangle(
    file_path=str(local_file_path),
    sheet_name=sheet_name,
    start_cell=start_cell,
    end_cell=end_cell,
    include_header_metadata=True
)

print(f"\nData shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")

if header_metadata:
    print(f"\nHeader metadata (row above headers):")
    for idx, val in header_metadata.items():
        print(f"  Column {idx}: {val}")

In [ ]:
# Preview the data
print("First 10 rows:")
display(df.head(10))

In [ ]:
# Data types
print("DataFrame dtypes:")
print(df.dtypes)

---
## Step 7: Profile Columns

Analyze each column to detect types, statistics, and quality issues.

In [ ]:
print("Profiling columns...")
print("=" * 60)

profiles = profile_columns(df, settings, header_metadata)

print(f"\nProfiled {len(profiles)} columns:\n")

# Summary table
profile_summary = []
for p in profiles:
    profile_summary.append({
        'Column': p.name,
        'Type': p.data_type,
        'Nulls': f"{p.null_count} ({p.null_percentage:.1f}%)",
        'Unique': p.unique_count,
        'Sample Values': str(p.sample_values[:3]) if p.sample_values else 'N/A',
    })

display(pd.DataFrame(profile_summary))

In [ ]:
# Detailed profile for each column
print("Detailed Column Profiles:")
print("=" * 60)

for p in profiles:
    print(f"\n[{p.name}]")
    print(f"  Type: {p.data_type}")
    print(f"  Null count: {p.null_count} ({p.null_percentage:.1f}%)")
    print(f"  Unique values: {p.unique_count}")
    
    if p.header_metadata:
        print(f"  Header metadata: {p.header_metadata}")
    
    if p.all_values:
        print(f"  All values (low cardinality): {p.all_values}")
    elif p.sample_values:
        print(f"  Sample values: {p.sample_values}")
    
    if p.statistics:
        print(f"  Statistics: {p.statistics}")
    
    if p.format_warnings:
        print(f"  Warnings: {p.format_warnings}")

---
## Step 8: Store Data (Parquet + Athena)

Save the data as Parquet to S3 and create/replace the Athena table.

In [ ]:
# table_id was set in Step 6 from selected_table
print(f"Table ID: {table_id}")
print(f"Storage mode: {settings.storage_mode}")
print("=" * 60)

# This will:
# - Save Parquet to S3 (or local)
# - DROP existing Athena table if exists
# - CREATE new Athena table
parquet_path, ddl = create_athena_table(df, table_id, profiles, settings)

print(f"\nParquet saved to: {parquet_path}")

if settings.storage_mode == 'aws':
    print(f"\nAthena DDL executed:")
    print("-" * 40)
    print(ddl)

---
## Step 9: LLM Generation

Generate descriptions, entities, and query patterns using the LLM.

**This section shows full visibility into:**
- Input prompts sent to LLM
- Raw LLM outputs
- Token usage

In [ ]:
# Import LLM components
from llm.bedrock_client import get_llm_client
from llm.description_generator import generate_description, format_column_info, format_sample_data, DESCRIPTION_SYSTEM_PROMPT
from llm.entity_extractor import extract_entities, format_for_entity_extraction, ENTITY_SYSTEM_PROMPT
from llm.pattern_generator import generate_query_patterns, format_for_pattern_generation, PATTERN_SYSTEM_PROMPT

# Get LLM client
llm_client = get_llm_client(settings)

# Prepare sample data for LLM
sample_df = df.head(settings.sample_rows_for_llm)

print(f"LLM Provider: {settings.llm_provider}")
print(f"Sample rows for LLM: {settings.sample_rows_for_llm}")
print(f"\nSample data preview:")
display(sample_df.head(5))

### 9a. Generate Table Description

In [ ]:
# Build the prompt (same logic as generate_description internally)
column_info = format_column_info(profiles)
sample_data = format_sample_data(df, settings.sample_rows_for_llm)

description_prompt = f"""Analyze this table extracted from Excel.

Source: {s3_key}, Sheet: "{sheet_name}", Range: {start_cell}:{end_cell or 'auto'}
Rows: {len(df)} (excluding header)
Columns: {len(df.columns)}

COLUMN PROFILES:
{column_info}

SAMPLE DATA (first {min(len(df), settings.sample_rows_for_llm)} rows):
{sample_data}

Based on this information, provide a description, purpose, and caveats for this table."""

print("SYSTEM PROMPT:")
print("=" * 60)
print(DESCRIPTION_SYSTEM_PROMPT)
print("\n" + "=" * 60)
print("USER PROMPT:")
print("=" * 60)
print(description_prompt)
print("=" * 60)

In [ ]:
# Call LLM
print("Calling LLM for table description...")
print("=" * 60)

description, purpose, caveats = generate_description(
    df=df,
    profiles=profiles,
    source_file=s3_key,
    source_sheet=sheet_name,
    source_range=f"{start_cell}:{end_cell or 'auto'}",
    client=llm_client,
    settings=settings,
)

print("\nLLM OUTPUT:")
print("-" * 40)
print(f"DESCRIPTION:\n{description}")
print(f"\nPURPOSE:\n{purpose}")
print(f"\nCAVEATS:\n{caveats}")

### 9b. Extract Entities

In [ ]:
# Build the prompt
entity_prompt = format_for_entity_extraction(df, profiles, description, purpose, caveats)

print("SYSTEM PROMPT:")
print("=" * 60)
print(ENTITY_SYSTEM_PROMPT)
print("\n" + "=" * 60)
print("USER PROMPT:")
print("=" * 60)
print(entity_prompt)
print("=" * 60)

In [ ]:
# Call LLM
print("Calling LLM for entity extraction...")
print("=" * 60)

entities, cross_refs = extract_entities(
    df=df,
    profiles=profiles,
    description=description,
    purpose=purpose,
    caveats=caveats,
    client=llm_client,
    settings=settings,
)

print("\nLLM OUTPUT:")
print("-" * 40)
print(f"Entities found: {len(entities)}")
for entity in entities:
    print(f"\n  [{entity.name}] {'(PRIMARY)' if entity.is_primary else ''}")
    print(f"    Identified by: {entity.identified_by}")
    print(f"    Cardinality: {entity.cardinality}")
    print(f"    Attributes: {entity.attributes[:5]}{'...' if len(entity.attributes) > 5 else ''}")
    print(f"    Description: {entity.description}")

print(f"\nCross-references found: {len(cross_refs)}")
for cr in cross_refs:
    print(f"  - {cr.column} -> {cr.likely_entity_type} ({cr.match_quality})")

### 9c. Generate Query Patterns

In [ ]:
# Build the prompt
pattern_prompt = format_for_pattern_generation(
    table_id=table_id,
    profiles=profiles,
    description=description,
    purpose=purpose,
    caveats=caveats,
    entities=entities,
)

print("SYSTEM PROMPT:")
print("=" * 60)
print(PATTERN_SYSTEM_PROMPT)
print("\n" + "=" * 60)
print("USER PROMPT:")
print("=" * 60)
print(pattern_prompt)
print("=" * 60)

In [ ]:
# Call LLM
print("Calling LLM for query patterns...")
print("=" * 60)

query_patterns = generate_query_patterns(
    table_id=table_id,
    profiles=profiles,
    description=description,
    purpose=purpose,
    caveats=caveats,
    entities=entities,
    client=llm_client,
    settings=settings,
)

print("\nLLM OUTPUT:")
print("-" * 40)
print(f"Query patterns generated: {len(query_patterns)}")
for i, pattern in enumerate(query_patterns, 1):
    print(f"\n  Pattern {i}: {pattern.natural_language}")
    print(f"    SQL: {pattern.sql[:100]}{'...' if len(pattern.sql) > 100 else ''}")
    if pattern.warnings:
        print(f"    Warning: {pattern.warnings}")

---
## Step 10: Assemble TableCard

Combine all the pieces into the final TableCard.

In [ ]:
# Build data quality summary
total_cells = df.shape[0] * df.shape[1]
null_cells = df.isnull().sum().sum()
all_warnings = []
for p in profiles:
    all_warnings.extend(p.format_warnings or [])

data_quality = DataQuality(
    row_count=df.shape[0],
    column_count=df.shape[1],
    null_percentage=round((null_cells / total_cells) * 100, 2) if total_cells > 0 else 0,
    format_warnings=list(set(all_warnings)),
)

# Build the TableCard
table_card = TableCard(
    table_id=table_id,
    source_file=s3_key,
    source_sheet=sheet_name,
    source_range=f"{start_cell}:{end_cell or 'auto'}",
    ingested_at=datetime.utcnow(),
    
    # Static half (from profiling)
    columns=profiles,
    data_quality=data_quality,
    
    # Dynamic half (from LLM)
    purpose=purpose,
    description=description,
    caveats=caveats,
    entities=entities,
    cross_references=cross_refs,
    query_patterns=query_patterns,
    
    # Storage info
    athena_database=settings.athena_database if settings.storage_mode == 'aws' else None,
    athena_table=table_id if settings.storage_mode == 'aws' else None,
    s3_location=parquet_path if settings.storage_mode == 'aws' else None,
    local_path=parquet_path if settings.storage_mode == 'local' else None,
)

print("TableCard assembled!")
print("=" * 60)
print(f"  Table ID: {table_card.table_id}")
print(f"  Purpose: {table_card.purpose}")
print(f"  Columns: {len(table_card.columns)}")
print(f"  Entities: {len(table_card.entities)}")
print(f"  Cross-refs: {len(table_card.cross_references)}")
print(f"  Query Patterns: {len(table_card.query_patterns)}")

---
## Step 11: Save TableCard

In [ ]:
# Serialize
card_json = serialize_card(table_card)

# Save
card_path = save_card(card_json, table_id, settings)

print(f"TableCard saved to: {card_path}")

In [ ]:
# Display full card JSON
print("Full TableCard JSON:")
print("=" * 60)
print(json.dumps(json.loads(card_json), indent=2))

---
## Step 12: Verify in Athena

Run a test query to verify the table was created correctly.

In [ ]:
if settings.storage_mode == 'aws':
    from ingestion.athena_loader import execute_athena_query
    
    # Run a simple count query
    test_query = f"SELECT COUNT(*) as row_count FROM {settings.athena_database}.{table_id}"
    print(f"Running test query: {test_query}")
    
    try:
        execution_id = execute_athena_query(test_query, settings)
        print(f"Query executed successfully! Execution ID: {execution_id}")
        
        # Get results
        athena = boto3.client('athena', region_name=settings.aws_region)
        results = athena.get_query_results(QueryExecutionId=execution_id)
        row_count = results['ResultSet']['Rows'][1]['Data'][0]['VarCharValue']
        print(f"\nTable {table_id} has {row_count} rows")
    except Exception as e:
        print(f"Query failed: {e}")
else:
    print("Skipping Athena verification (local mode)")
    print(f"\nLocal parquet file: {parquet_path}")
    
    # Verify local parquet
    import pyarrow.parquet as pq
    pq_table = pq.read_table(parquet_path)
    print(f"Parquet file has {pq_table.num_rows} rows")

---
## Step 13: Save Configuration (Optional)

If you configured this file manually, save the configuration so you don't have to re-enter coordinates next time.

In [ ]:
# Uncomment to save this configuration

# new_config_name = "my_new_table"  # Change this
# table_configs[new_config_name] = selected_config
# save_table_configs(table_configs)
# print(f"Configuration saved as '{new_config_name}'")

---
## Summary

Ingestion complete!

In [ ]:
print("=" * 60)
print("INGESTION SUMMARY")
print("=" * 60)
print(f"Source: s3://{settings.s3_bucket_raw}/{s3_key}")
print(f"Sheet: {sheet_name}")
print(f"Range: {start_cell} -> {end_cell or 'auto-detected'}")
print(f"")
print(f"Table ID: {table_id}")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print(f"")
print(f"Parquet: {parquet_path}")
print(f"Card: {card_path}")
if settings.storage_mode == 'aws':
    print(f"Athena: {settings.athena_database}.{table_id}")
print("=" * 60)

---

## Batch Processing: Ingest All Configured Files

Run this section to process all files defined in `table_configs.yaml`.

In [ ]:
# UNCOMMENT TO RUN BATCH PROCESSING

# from ingestion.card_builder import build_card
# 
# for config_name, config in table_configs.items():
#     print(f"\n{'='*60}")
#     print(f"Processing: {config_name}")
#     print(f"{'='*60}")
#     
#     try:
#         # Download file
#         file_s3_key = config['s3_key']
#         local_path = local_temp_dir / file_s3_key.split('/')[-1]
#         s3.download_file(settings.s3_bucket_raw, file_s3_key, str(local_path))
#         
#         # Process each table (can be multiple per sheet)
#         for table_cfg in config.get('tables', []):
#             card = build_card(
#                 file_path=str(local_path),
#                 sheet_name=table_cfg['sheet'],
#                 start_cell=table_cfg.get('start_cell', 'A1'),
#                 end_cell=table_cfg.get('end_cell'),
#                 table_id=table_cfg.get('table_id'),
#                 settings=settings,
#             )
#             print(f"  Created: {card.table_id} (sheet: {table_cfg['sheet']})")
#     except Exception as e:
#         print(f"  ERROR: {e}")